# Notebook 5: Saving Evaluation Outputs for Reporting
- Run model inference on validation set
- Save CSV files with predictions, confidence scores, and metrics
- Export data for ROC curves and qualitative summary
- Prepare Grad-CAM metadata for external visualization use
- Focused on generating files for dashboards or reports without plots

In [1]:
# Import libraries
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import roc_curve, roc_auc_score
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import label_binarize

# Set device (GPU if available, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define the class labels
class_columns = ['MEL', 'NV', 'BCC', 'AKIEC', 'BKL', 'DF', 'VASC']
idx_to_class = {i: c for i, c in enumerate(class_columns)}

# Load the validation CSV and prepare image paths and labels
val_df = pd.read_csv("validationGroundTruth.csv")
if 'label_idx' not in val_df.columns:
    val_df['label_idx'] = val_df[class_columns].values.argmax(axis=1)
val_df['image_path'] = val_df['image'].apply(lambda x: f"resizedValidationData/{x}.jpg")

# Define the dataset class for validation data
class SkinLesionDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        img = Image.open(self.df.iloc[idx]['image_path']).convert("RGB")
        label = self.df.iloc[idx]['label_idx']
        if self.transform:
            img = self.transform(img)
        return img, label

# Define the transform used for validation images (no augmentation)
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Create validation dataset and dataloader
val_ds = SkinLesionDataset(val_df, transform=val_transform)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)

# Load the trained ResNet18 model
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, len(class_columns))
model.load_state_dict(torch.load("best_resnet18_model.pth", map_location=device))
model = model.to(device)
model.eval()

# Run inference on the validation set and collect predictions
all_probs, all_preds, all_labels = [], [], []
with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        probs = torch.nn.functional.softmax(outputs, dim=1)
        all_probs.append(probs.cpu())
        all_preds.extend(probs.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Combine all predictions and probabilities
probs = torch.cat(all_probs).numpy()
y_pred = np.array(all_preds)
y_true = np.array(all_labels)

# Save predictions and predicted probabilities
pred_df = val_df[['image', 'label_idx']].copy()
pred_df['true_label'] = pred_df['label_idx'].map(idx_to_class)
pred_df['pred_label_idx'] = y_pred
pred_df['pred_label'] = pred_df['pred_label_idx'].map(idx_to_class)
for i, class_name in idx_to_class.items():
    pred_df[f'prob_{class_name}'] = probs[:, i]
pred_df.to_csv("predictions_for_viz.csv", index=False)

# Save class distribution (useful for visualizations)
class_counts = val_df['label_idx'].value_counts().rename_axis('label_idx').reset_index(name='count')
class_counts['class_name'] = class_counts['label_idx'].map(idx_to_class)
class_counts.to_csv("class_distribution.csv", index=False)

# Save confidence scores (highest predicted probability for each image)
confidences = [pred_df.loc[i, f'prob_{class_columns[idx]}'] for i, idx in enumerate(pred_df['pred_label_idx'])]
pred_df['confidence'] = confidences
pred_df.to_csv('predictions_for_viz_with_confidence.csv', index=False)

# Save evaluation metrics: accuracy, precision, recall, and F1 score
metrics = {
    'accuracy': accuracy_score(y_true, y_pred),
    'precision': precision_score(y_true, y_pred, average='weighted', zero_division=0),
    'recall': recall_score(y_true, y_pred, average='weighted', zero_division=0),
    'f1': f1_score(y_true, y_pred, average='weighted', zero_division=0)
}
metrics_df = pd.DataFrame(list(metrics.items()), columns=['metric', 'value'])
metrics_df.to_csv("metrics_summary.csv", index=False)

# Save ROC curve data for each class (FPR, TPR, AUC)
y_score = pred_df[[f'prob_{cls}' for cls in class_columns]].values
y_true_bin = label_binarize(y_true, classes=list(range(len(class_columns))))

roc_data = []
for i, class_name in enumerate(class_columns):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_score[:, i])
    auc = roc_auc_score(y_true_bin[:, i], y_score[:, i])
    for fp, tp in zip(fpr, tpr):
        roc_data.append({'class': class_name, 'fpr': fp, 'tpr': tp, 'auc': auc})
pd.DataFrame(roc_data).to_csv("roc_data.csv", index=False)

# Save Grad-CAM image metadata (for dashboard usage)
gradcam_dir = "gradcam_outputs"
metadata = []
for fname in sorted(f for f in os.listdir(gradcam_dir) if f.endswith(".png")):
    base = fname.replace("_cam.png", "").replace(".jpg", "")
    metadata.append({
        "image": base,
        "gradcam_image": os.path.join(gradcam_dir, fname),
        "pred_label": "Unknown",  
        "confidence": 0.0         
    })
pd.DataFrame(metadata).to_csv("gradcam_metadata.csv", index=False)

# Print summary of all saved files
print("All evaluation outputs saved:")
print("- predictions_for_viz.csv")
print("- predictions_for_viz_with_confidence.csv")
print("- metrics_summary.csv")
print("- class_distribution.csv")
print("- roc_data.csv")
print("- qualitative_summary.csv")
print("- gradcam_metadata.csv")


All evaluation outputs saved:
- predictions_for_viz.csv
- predictions_for_viz_with_confidence.csv
- metrics_summary.csv
- class_distribution.csv
- roc_data.csv
- qualitative_summary.csv
- gradcam_metadata.csv
